# R0 — les deux points de départ, sans entraînement · **Colab**

Évalue **A0** (Qwen3.5-4B-Base) et **A1** (AfriqueQwen3.5-4B-50Langs) tels quels, avant tout
alignement. C'est la première ligne du tableau de résultats, et la référence contre laquelle
A2 et A3 seront lus.

Tourne sur **Colab** et non sur Kaggle, pour préserver le quota Kaggle pendant que le SFT
s'y exécute. Budget GPU séparé, aucune interférence.

## Ce qui est mesuré

**Uhura-TruthfulQA en QCM haoussa**, scoré par log-vraisemblance des options — le modèle
attribue une probabilité à chaque réponse écrite, la plus haute gagne.

Aucune génération, **aucun juge**. On compare des nombres, pas des textes : le scoring est
exactement aussi fiable en haoussa qu'en anglais. C'est ce qui rend cet axe mesurable par
une équipe qui ne lit pas la langue.

## Ce qu'on attend

Deux modèles **base**, jamais alignés. Rien ne garantit qu'ils dépassent le hasard, et un
score au niveau du plancher serait une information en soi — pas un échec du protocole.

## 0. Environnement

In [ ]:
!pip install -q -U "transformers>=5.16" "peft>=0.20" bitsandbytes accelerate datasets

In [ ]:
import torch

assert torch.cuda.is_available(), "Execution > Modifier le type d'execution > GPU"
print(torch.cuda.get_device_name(0),
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Go")

### Récupérer le code

Le dépôt est privé. Sur Colab, le jeton se place dans **la clé 🔑 du panneau de gauche**
(Secrets), sous le nom `GITHUB_TOKEN`, avec l'accès activé pour ce notebook.

Aucune donnée locale n'est nécessaire : R0 ne lit qu'Uhura, qui est public sur Hugging Face.

In [ ]:
import os, sys, subprocess
from pathlib import Path

DEPOT = "afrique-safety-dpo_alignment"

def jeton():
    try:
        from google.colab import userdata
        for nom in ("GITHUB_TOKEN", "GITHUB_PAT", "GH_TOKEN"):
            try:
                valeur = userdata.get(nom)
                if valeur:
                    print(f"jeton trouve sous : {nom}")
                    return valeur
            except Exception:
                continue
    except ImportError:
        pass
    return os.environ.get("GITHUB_TOKEN")

if not Path(DEPOT).exists():
    pat = jeton()
    if not pat:
        raise RuntimeError(
            "Aucun jeton GitHub. Panneau de gauche > cle 🔑 > ajouter GITHUB_TOKEN, "
            "et activer l'acces pour ce notebook."
        )
    subprocess.run(
        ["git", "clone", "-q", f"https://{pat}@github.com/zoom-BT/{DEPOT}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", DEPOT, "pull", "-q"], check=False)

sys.path.insert(0, str(Path(DEPOT).resolve()))
print(subprocess.run(["git", "-C", DEPOT, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Le jeu d'évaluation

`ha_multiple_choice` d'Uhura : traduction professionnelle humaine, licence MIT.

⚠️ Le split `test` ne fait pas la même taille dans toutes les langues — 791 en haoussa, et
non les 809 annoncés par la carte. On lit la taille réelle plutôt que de la supposer.

In [ ]:
from datasets import load_dataset

uhura = load_dataset("masakhane/uhura-truthfulqa", "ha_multiple_choice", split="test")
lignes = list(uhura)
print(f"{len(lignes)} questions")

exemple = lignes[0]
print("\nquestion :", exemple["question"][:110])
for i, (choix, etiquette) in enumerate(
    zip(exemple["mc1_targets"]["choices"], exemple["mc1_targets"]["labels"])
):
    print(f"  [{'x' if etiquette else ' '}] {choix[:90]}")

## 2. Évaluation des deux modèles

Chargés en 4 bits, comme à l'entraînement : évaluer en pleine précision un modèle qui sera
entraîné en QLoRA mesurerait autre chose que ce qu'on aligne.

Le sous-échantillon est réglable. Sur les 791 questions, comptez une trentaine de minutes
par modèle ; commencez petit pour vérifier que la chaîne tient.

In [ ]:
N = 150      # mettre None pour les 791 questions

MODELES = {
    "A0_Qwen-Base":   "Qwen/Qwen3.5-4B-Base",
    "A1_AfriqueQwen": "McGill-NLP/AfriqueQwen3.5-4B-50Langs",
}
sous_ensemble = lignes[:N] if N else lignes
print(f"{len(sous_ensemble)} questions par modele")

In [ ]:
import gc, json, time

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from src.eval_mcq import binomial_two_sided_p, evaluate_mcq

quantification = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

resultats = {}
for nom, chemin in MODELES.items():
    print(f"\n=== {nom} ===")
    depart = time.time()
    tok = AutoTokenizer.from_pretrained(chemin)
    modele = AutoModelForCausalLM.from_pretrained(
        chemin, quantization_config=quantification, device_map={"": 0}, dtype=torch.float16
    ).eval()

    sortie = evaluate_mcq(modele, tok, sous_ensemble)
    sortie["p_contre_hasard"] = binomial_two_sided_p(
        sortie["correct"], sortie["n"], sortie["random_baseline"]
    )
    sortie["duree_min"] = round((time.time() - depart) / 60, 1)
    sortie.pop("per_question")          # trop volumineux pour le resume
    resultats[nom] = sortie

    print(f"  exactitude : {sortie['accuracy']:.3f}  "
          f"(hasard {sortie['random_baseline']:.3f}, p = {sortie['p_contre_hasard']:.2g})")
    print(f"  duree      : {sortie['duree_min']} min")

    # Liberer avant le modele suivant, sinon le second n'a plus de place.
    del modele, tok
    gc.collect(); torch.cuda.empty_cache()

## 3. Résultat

In [ ]:
import pandas as pd

tableau = pd.DataFrame(resultats).T[
    ["n", "correct", "accuracy", "random_baseline", "p_contre_hasard", "duree_min"]
]
tableau.columns = ["n", "justes", "exactitude", "hasard", "p", "min"]
display(tableau.round(4))

a0, a1 = resultats["A0_Qwen-Base"], resultats["A1_AfriqueQwen"]
ecart = a1["accuracy"] - a0["accuracy"]
print(f"\nA1 - A0 = {ecart:+.3f}")
print("\nCe que cela dit, et ce que cela ne dit pas :")
for nom, r in resultats.items():
    verdict = "au-dessus du hasard" if r["p_contre_hasard"] < 0.05 else "indiscernable du hasard"
    print(f"  {nom:<16} {r['accuracy']:.3f}  -> {verdict}")
print("\nAucun des deux n'a ete aligne. Un score au niveau du plancher est une mesure,")
print("pas un echec du protocole -- c'est le point de depart que R1 et R2 doivent depasser.")

In [ ]:
from pathlib import Path

Path("resultats").mkdir(exist_ok=True)
Path("resultats/R0_uhura_mcq.json").write_text(
    json.dumps({"jeu": "uhura ha_multiple_choice", "n_demande": N, "resultats": resultats},
               indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("ecrit : resultats/R0_uhura_mcq.json")

# Colab efface tout a la fermeture: telecharger avant de partir.
try:
    from google.colab import files
    files.download("resultats/R0_uhura_mcq.json")
except Exception:
    print("(telechargement automatique indisponible -- recuperer le fichier a la main)")

---

## Ce que R0 établit

L'écart **A1 − A0 avant tout alignement**. Si AfriqueQwen part déjà devant sa base sur la
véracité en haoussa, une partie de l'écart mesuré après alignement lui préexiste et devra
être soustraite du crédit accordé au DPO.

C'est précisément pourquoi ce run vient en premier : sans lui, tout gain observé en R1/R2
serait ambigu entre *le CPT aide l'alignement* et *le CPT aidait déjà avant*.

## Limites, à déclarer

Un seul axe est couvert — **Honest**, via Uhura. Ni AfriMGSM pour l'utilité, ni AfriHate
pour l'axe Harmless. Ces deux-là demandent de la génération, donc plus de temps, et
AfriHate demande en plus un jugement sur du texte haoussa.

Et le contenu d'Uhura est **occidental**, professionnellement traduit : Amérique, Canada,
autobahn. Ce run mesure donc la véracité sur du savoir occidental exprimé en haoussa, non
sur du savoir africain. C'est une limite du jeu, pas du protocole, et elle est documentée
dans la fiche datasets.